In [4]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# Charger le fichier CSV
df = pd.read_csv("collecte_de_données.csv")  # change le nom de ton fichier ici si besoin

# Colonnes concernées par l’imputation
colonnes_cibles = [
    "UC", "SER", "PLAT", "CRT", "IMP", "Fax", "Photocop", "Onduleurs", 
    "Pc portable", "Tel", "GSM", "Claviers", "Souris", "Switch", 
    "Routeur", "V-Projecteur", "Cable", "Tonner", "Scanner", "Divers"
]

# Remplacer les 0 par NaN pour les colonnes à imputer
df_cibles = df[colonnes_cibles].replace(0, np.nan)

# Trouver les colonnes entièrement vides
colonnes_vides = df_cibles.columns[df_cibles.isna().all()].tolist()

# Garder uniquement les colonnes avec au moins une valeur connue
colonnes_a_imputer = [col for col in colonnes_cibles if col not in colonnes_vides]

# Imputation avec IterativeImputer + RandomForest
imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=10, random_state=42),
    max_iter=10,
    random_state=42
)
imputed_array = imputer.fit_transform(df_cibles[colonnes_a_imputer])
df_imputed = pd.DataFrame(imputed_array, columns=colonnes_a_imputer)

# Arrondir les valeurs et convertir en entiers
df_imputed = df_imputed.round().astype(int)

# Ajouter les colonnes entièrement vides (remplies de 0 ou NaN)
for col in colonnes_vides:
    df_imputed[col] = 0  # ou np.nan si tu préfères garder vide

# Réorganiser les colonnes dans l’ordre initial
df_imputed = df_imputed[colonnes_cibles]

# Fusion avec les autres colonnes du DataFrame original
df_autres = df.drop(columns=colonnes_cibles)
df_final = pd.concat([df_autres.reset_index(drop=True), df_imputed.reset_index(drop=True)], axis=1)

# Sauvegarder dans un nouveau fichier CSV
df_final.to_csv("C:/Users/HP G3/Bureau/Collecte_données_imputé.csv", index=False, encoding='utf-8-sig')

# Afficher le résultat
print("✅ Données imputées et converties en entiers avec succès !")
print(df_final.head())


✅ Données imputées et converties en entiers avec succès !
         Date   Partenaire  UC  SER  PLAT  CRT  IMP  Fax  Photocop  Onduleurs  \
0  2024-01-04  BANK ASSAFA  65   15    95   11   18    2         2         31   
1  2024-01-05  BANK ASSAFA  81   27   107   11    2    1         1         25   
2  2024-01-09        2 WLS  10   22     8    8   18    1         1         27   
3  2024-01-10    MEDIATING  25   30    10   10    2    1         1         27   
4  2024-01-12      AW BANK  31   37   124   12   10    2         1         28   

   ...  GSM  Claviers  Souris  Switch  Routeur  V-Projecteur  Cable  Tonner  \
0  ...   25        67      61       5       57             1      0       0   
1  ...   18        65      72      15       98             1      0       0   
2  ...   16         8      44      28       84             1      0       0   
3  ...   16        22      33      34       98             1      0       0   
4  ...   13        21      26      53       98             1

C:\Python\Python312\Lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [1]:
import pandas as pd

# Chargement du fichier
df = pd.read_csv("C:/Users/HP G3/Bureau/Collecte_données_imputé.csv")

# Détection des colonnes numériques
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Fonction pour détecter les outliers selon l'IQR
def has_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return len(outliers) > 0

# Liste des colonnes contenant des outliers
cols_with_outliers = [col for col in numeric_cols if has_outliers(df[col])]

# Affichage des résultats
print("Colonnes contenant des outliers :")
for col in cols_with_outliers:
    print(f"- {col}")


Colonnes contenant des outliers :
- UC
- IMP
- Fax
- Onduleurs
- GSM
- Switch
- Scanner
- Divers


In [4]:
import pandas as pd
import numpy as np

# Chargement du fichier
df = pd.read_csv("C:/Users/HP G3/Bureau/Collecte_données_imputé.csv")

# Liste des colonnes à transformer
cols_to_log_transform = ['UC', 'IMP', 'Fax', 'Onduleurs', 'GSM', 'Switch', 'Scanner', 'Divers']

# Application du log(x + 1) à toutes les valeurs de ces colonnes
for col in cols_to_log_transform:
    df[col] = np.round(np.log1p(df[col].astype(float)), 2)

# Sauvegarde du fichier modifié
output_path = "C:/Users/HP G3/Bureau/Collecte_donnees_log_all.csv"
df.to_csv(output_path, index=False)

print(f"✅ Fichier corrigé avec succès : {output_path}")


✅ Fichier corrigé avec succès : C:/Users/HP G3/Bureau/Collecte_donnees_log_all.csv


In [5]:
import pandas as pd
import numpy as np
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objs as go
import plotly.express as px

# Chargement des fichiers
df_original = pd.read_csv("C:/Users/HP G3/Bureau/Collecte_données_imputé.csv")
df_transformed = pd.read_csv("C:/Users/HP G3/Bureau/Collecte_donnees_log_all.csv")

# Détection de toutes les colonnes numériques
numeric_columns = df_original.select_dtypes(include=[np.number]).columns.tolist()

# Calcul de la matrice de corrélation après traitement des outliers
df_transformed_numeric = df_transformed.select_dtypes(include=[np.number])
correlation_matrix = df_transformed_numeric.corr()

# Initialisation de l'app
app = dash.Dash(__name__)
app.title = "Analyse complète des colonnes numériques"

# Layout
app.layout = html.Div([
    html.H1("Analyse des Colonnes Numériques (Avant / Après)", style={'textAlign': 'center'}),

    html.Label("Sélectionner une colonne :"),
    dcc.Dropdown(
        id='column-dropdown',
        options=[{'label': col, 'value': col} for col in numeric_columns],
        value=numeric_columns[0]
    ),

    html.Div([
        # Histogrammes dans une ligne
        html.Div([
            dcc.Graph(id='histogram-avant-graph', style={'width': '48%', 'display': 'inline-block'}),
            dcc.Graph(id='histogram-apres-graph', style={'width': '48%', 'display': 'inline-block'})
        ], style={'display': 'flex', 'justify-content': 'space-between'}),

        # Boxplot Avant et Après
        dcc.Graph(id='boxplot-graph'),

        # Matrice de Corrélation (Heatmap)
        dcc.Graph(id='correlation-heatmap')
    ])
])

# Callback
@app.callback(
    [Output('histogram-avant-graph', 'figure'),
     Output('histogram-apres-graph', 'figure'),
     Output('boxplot-graph', 'figure'),
     Output('correlation-heatmap', 'figure')],
    [Input('column-dropdown', 'value')]
)
def update_graphs(column):
    # Histogrammes Avant
    hist_avant_fig = go.Figure()
    hist_avant_fig.add_trace(go.Histogram(
        x=df_original[column],
        name='Avant',
        opacity=0.6,
        marker_color='#ff7f0e'  # Orange
    ))
    hist_avant_fig.update_layout(
        title=f"Distribution de '{column}' - Avant",
        xaxis_title='Valeur',
        yaxis_title='Fréquence'
    )

    # Histogrammes Après
    hist_apres_fig = go.Figure()
    hist_apres_fig.add_trace(go.Histogram(
        x=df_transformed[column],
        name='Après',
        opacity=0.6,
        marker_color='#1f77b4'  # Jaune
    ))
    hist_apres_fig.update_layout(
        title=f"Distribution de '{column}' - Après (log si outlier)",
        xaxis_title='Valeur',
        yaxis_title='Fréquence'
    )

    # Boxplot Avant et Après
    box_fig = go.Figure()
    box_fig.add_trace(go.Box(
        y=df_original[column],
        name='Avant',
        marker_color='#ff7f0e'  # Orange
    ))
    box_fig.add_trace(go.Box(
        y=df_transformed[column],
        name='Après',
        marker_color='#1f77b4'  # Jaune
    ))
    box_fig.update_layout(
        title=f"Boxplot de '{column}'",
        yaxis_title='Valeur'
    )

    # Matrice de corrélation (Heatmap) avec annotations
    heatmap_fig = px.imshow(correlation_matrix, 
                            color_continuous_scale='RdBu_r', 
                            title="Matrice de Corrélation des Equipements",
                            text_auto=True)  # Ajout des coefficients de corrélation comme annotations
    
    # Mise à jour de la taille du graphique (plus grand)
    heatmap_fig.update_layout(
        height=900,  # Hauteur du graphique
        width=1300,  # Largeur du graphique
        xaxis_title='Colonnes',
        yaxis_title='Colonnes',
        coloraxis_colorbar_title="Corrélation",
        margin=dict(t=50, b=50, l=50, r=50)  # Marges personnalisées
    )

    return hist_avant_fig, hist_apres_fig, box_fig, heatmap_fig

# Exécution de l'application
if __name__ == '__main__':
    app.run(debug=True)
